## Pre-processing Objectives:
1. Create a measure for recidivism in the entire dataset: try various approaches to this as different columns
2. Check the distribution of missing values across states
3. Clean up missing values specific to states chosen

In [1]:
import warnings

# warnings.filterwarnings('ignore')

import pandas as pd

pd.set_option("mode.copy_on_write", True)

In [2]:
# load data
import os
import gdown

file_id = "1jkBkjOsdr-0hhUgj3p2J7pk1Welkk7A1"

# check if a csv was already created earlier and download only if it wasn't
if not os.path.exists("dataset.csv"):
    gdown.download(id=file_id, output="dataset.csv", quiet=False)

df = pd.read_csv("dataset.csv", sep="\t")
df.head()

,ABT_INMATE_ID,SEX,ADMTYPE,OFFGENERAL,EDUCATION,ADMITYR,RELEASEYR,MAND_PRISREL_YEAR,PROJ_PRISREL_YEAR,PARELIG_YEAR,SENTLGTH,OFFDETAIL,RACE,AGEADMIT,AGERELEASE,TIMESRVD,RELTYPE,STATE
0,A012021000000090128,1,1,2,9,2013,2013,9999,9999,9999,4,8,1,5,5,0,3,1
1,A012021000000110168,1,1,1,9,2004,9999,9999,9999,9999,4,3,1,5,9,9,9,1
2,A012021000000090187,1,1,2,9,2009,2009,9999,9999,9999,0,11,9,1,1,0,1,1
3,A012021000000010425,1,3,3,9,2016,2016,9999,9999,9999,2,12,9,3,3,0,3,1
4,A012021000000090117,1,1,1,9,2009,2017,9999,9999,9999,4,4,9,1,2,3,1,1


In [3]:
df = df.drop_duplicates()
df.shape

(13768054, 18)

In [ ]:
# df = df.drop(df["ADMITYR"] == None)
df["ADMITYR"].value_counts()

ADMITYR
2008    640278
2009    638225
2007    624535
2010    616765
2006    610434
         ...  
1952        29
1953        21
1951        21
1955        20
1950        16
Name: count, Length: 72, dtype: int64

In [4]:
# add state names
state_map = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
}

df["STATE_NAME"] = df["STATE"].map(state_map)

# Verify
print(df["STATE_NAME"].unique())

['Alabama' 'Arizona' 'California' 'Colorado' 'Delaware' 'Florida'
 'Georgia' 'Illinois' 'Indiana' 'Iowa' 'Kansas' 'Kentucky' 'Louisiana'
 'Maine' 'Maryland' 'Massachusetts' 'Michigan' 'Minnesota' 'Mississippi'
 'Missouri' 'Montana' 'Nebraska' 'Nevada' 'New Hampshire' 'New Jersey'
 'New Mexico' 'New York' 'North Carolina' 'North Dakota' 'Ohio' 'Oklahoma'
 'Oregon' 'Pennsylvania' 'Rhode Island' 'South Carolina' 'South Dakota'
 'Tennessee' 'Texas' 'Utah' 'Washington' 'West Virginia' 'Wisconsin' nan
 'District of Columbia']


1. Create a measure for recidivism in the entire dataset: try various approaches to this as different columns

In [5]:
df["tot_arrest_counts"] = df.groupby("ABT_INMATE_ID")["ABT_INMATE_ID"].transform(
    "count"
)
df

,ABT_INMATE_ID,SEX,ADMTYPE,OFFGENERAL,EDUCATION,ADMITYR,RELEASEYR,MAND_PRISREL_YEAR,PROJ_PRISREL_YEAR,PARELIG_YEAR,SENTLGTH,OFFDETAIL,RACE,AGEADMIT,AGERELEASE,TIMESRVD,RELTYPE,STATE,STATE_NAME,tot_arrest_counts
0,A012021000000090128,1,1,2,9,2013,2013,9999,9999,9999,4,8,1,5,5,0,3,1,Alabama,1
1,A012021000000110168,1,1,1,9,2004,9999,9999,9999,9999,4,3,1,5,9,9,9,1,Alabama,1
2,A012021000000090187,1,1,2,9,2009,2009,9999,9999,9999,0,11,9,1,1,0,1,1,Alabama,1
3,A012021000000010425,1,3,3,9,2016,2016,9999,9999,9999,2,12,9,3,3,0,3,1,Alabama,4
4,A012021000000090117,1,1,1,9,2009,2017,9999,9999,9999,4,4,9,1,2,3,1,1,Alabama,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13897851,A112020000001771624,1,1,3,9,2007,2010,2011,2010,9999,2,12,2,9,9,2,1,11,District of Columbia,1
13897852,A112020000001771626,1,1,1,9,2001,2004,2004,2004,9999,3,4,2,9,9,2,1,11,District of Columbia,2
13897853,A112020000001771626,1,1,1,9,2007,2013,2017,2017,9999,4,4,2,9,9,3,1,11,District of Columbia,2
13897854,A112020000001771666,1,1,1,9,2010,2014,2014,2013,9999,2,1,2,9,9,2,1,11,District of Columbia,2


In [ ]:
# Step 1: Get the latest release year per inmate per year
latest_release = (
    df.groupby(["ABT_INMATE_ID", "ADMITYR"])["RELEASEYR"]
    .max()
    .reset_index()
    .rename(columns={"RELEASEYR": "LATEST_RELEASEYR"})
)

# Step 2: Get unique inmate-year combinations for future appearances
future_appearances = df[["ABT_INMATE_ID", "ADMITYR"]].drop_duplicates()

# Step 3: Self-merge to compare each inmate's latest release year against future admissions
merged = latest_release.merge(
    future_appearances, on="ABT_INMATE_ID", suffixes=("", "_FUTURE")
)

# Step 4: Check if future admission falls within 2 years after latest release
recid_flag = (
    merged[
        (merged["ADMITYR_FUTURE"] > merged["LATEST_RELEASEYR"])
        & (merged["ADMITYR_FUTURE"] <= merged["LATEST_RELEASEYR"] + 2)
    ][["ABT_INMATE_ID", "ADMITYR"]]
    .drop_duplicates()
    .assign(within_2_yrs=1)
)

# Step 5: Merge back and fill non-recidivists with 0
df = df.merge(recid_flag, on=["ABT_INMATE_ID", "ADMITYR"], how="left")
df["within_2_yrs"] = df["within_2_yrs"].fillna(0).astype(int)

In [7]:
df

,ABT_INMATE_ID,SEX,ADMTYPE,OFFGENERAL,EDUCATION,ADMITYR,RELEASEYR,MAND_PRISREL_YEAR,PROJ_PRISREL_YEAR,PARELIG_YEAR,...,OFFDETAIL,RACE,AGEADMIT,AGERELEASE,TIMESRVD,RELTYPE,STATE,STATE_NAME,tot_arrest_counts,within_2_yrs
0,A012021000000090128,1,1,2,9,2013,2013,9999,9999,9999,...,8,1,5,5,0,3,1,Alabama,1,0
1,A012021000000110168,1,1,1,9,2004,9999,9999,9999,9999,...,3,1,5,9,9,9,1,Alabama,1,0
2,A012021000000090187,1,1,2,9,2009,2009,9999,9999,9999,...,11,9,1,1,0,1,1,Alabama,1,0
3,A012021000000010425,1,3,3,9,2016,2016,9999,9999,9999,...,12,9,3,3,0,3,1,Alabama,4,1
4,A012021000000090117,1,1,1,9,2009,2017,9999,9999,9999,...,4,9,1,2,3,1,1,Alabama,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13768049,A112020000001771624,1,1,3,9,2007,2010,2011,2010,9999,...,12,2,9,9,2,1,11,District of Columbia,1,0
13768050,A112020000001771626,1,1,1,9,2001,2004,2004,2004,9999,...,4,2,9,9,2,1,11,District of Columbia,2,0
13768051,A112020000001771626,1,1,1,9,2007,2013,2017,2017,9999,...,4,2,9,9,3,1,11,District of Columbia,2,0
13768052,A112020000001771666,1,1,1,9,2010,2014,2014,2013,9999,...,1,2,9,9,2,1,11,District of Columbia,2,1


2. Check the distribution of missing values across states

3. Clean up missing values specific to states chosen